# Load-out-of-store vs direct-path timing

How much does going *through the store* cost versus just reading a file that is already sitting at a known path? This times **every step** of retrieving an artifact through the API —

1. **finding** the node (by id, and by metadata search),
2. **discovering** its artifact with `.artifacts()` (which, for a chunked store, also *reassembles* the file from the chunk pool),
3. **reading / parsing** the returned path,

— and compares the total against `pd.read_csv(path)` on a plain copy of the same file. Because reads route through a session cache, the store path has two regimes: **cold** (first touch — decompress from the pool) and **warm** (cache hit — a plain file read). Both are measured.

Re-run top to bottom; the stores are wiped at the start.

In [1]:
import shutil
import time
from pathlib import Path
import numpy as np
import pandas as pd
import ancestree

ROOT = Path("load_benchmark_store")  # the lineage store (chunk=True)
BASE = Path("load_benchmark_direct")  # plain copies of the same files, no store
for d in (ROOT, BASE):
    if d.exists():
        shutil.rmtree(d)
BASE.mkdir(parents=True)

store = ancestree.LineageStore(ROOT, dedupe=True, chunk=True)


def timed(label, fn, reps=50):
    ts = []
    for _ in range(reps):
        t = time.perf_counter()
        fn()
        ts.append((time.perf_counter() - t) * 1000)
    print(f"{label:<44} median {np.median(ts):8.3f} ms   (best {min(ts):.3f})")
    return float(np.median(ts))

## 1. Build a store with real artifacts (+ plain copies for the baseline)

Most nodes are metadata-only (to give the index a realistic size for the search step); a handful hold a ~2 MB CSV. Each artifact is also written verbatim to `BASE/` so we can read the *identical bytes* directly, with no store involved.

In [2]:
N_TOTAL = 1000  # nodes in the store (search scans the whole index)
N_ART = 30  # of those, how many hold a real artifact we will load
ROWS = 95_000  # ~2 MB per CSV
rng = np.random.default_rng(0)


def make_csv(nrows):
    return pd.DataFrame(
        {
            "ts": np.arange(nrows),
            "a": rng.normal(size=nrows).round(4),
            "b": rng.normal(size=nrows).round(4),
            "label": rng.integers(0, 3, nrows),
        }
    )


target_ids, direct_paths = [], []
t0 = time.perf_counter()
for i in range(N_TOTAL):
    with store.create_node(step_type="run") as n:
        n.add_meta("run_id", i)
        n.add_meta("accuracy", round(float(rng.random()), 4))
        if i < N_ART:
            df = make_csv(ROWS)
            df.to_csv(n / "data.csv", index=False)  # stored (chunked)
            p = BASE / f"run_{i}.csv"
            df.to_csv(p, index=False)  # plain copy
            target_ids.append(n.node_id)
            direct_paths.append(p)

art_mb = direct_paths[0].stat().st_size / 1e6
print(
    f"built {N_TOTAL} nodes ({N_ART} with ~{art_mb:.1f} MB CSVs) in {time.perf_counter() - t0:.0f}s"
)
print(
    "node dir of a target holds only:",
    sorted(p.name for p in (ROOT / target_ids[0]).iterdir()),
)

built 1000 nodes (30 with ~2.2 MB CSVs) in 28s
node dir of a target holds only: ['.artifacts.json', 'data.csv', 'meta.json']


## 2. Baseline — the file is already at a known path

No store, no lineage, no decompression: just read the bytes / parse the CSV sitting on disk. This is the bar everything else is compared against.

In [3]:
direct_bytes = timed("direct  path.read_bytes()", lambda: direct_paths[0].read_bytes())
direct_parse = timed("direct  pd.read_csv(path)", lambda: pd.read_csv(direct_paths[0]))

direct  path.read_bytes()                    median    0.075 ms   (best 0.062)
direct  pd.read_csv(path)                    median   12.178 ms   (best 12.009)


## 3. Retrieve through the store, by known id — every step

You already know the node id. Time each step: `get_node` (loads `meta.json`), `.artifacts()` (discovers the file and, on a cold cache, reassembles it from chunks), then `pd.read_csv` on the returned path. Cold clears the cache first so every artifact is a fresh miss; warm runs straight after, all cache hits.

In [4]:
def breakdown_by_id(clear):
    if clear:
        store.clear_cache()
    g = a = r = 0.0
    for tid in target_ids:
        t = time.perf_counter()
        node = store.get_node(tid)
        g += time.perf_counter() - t
        t = time.perf_counter()
        path = node.artifacts("data.csv")[0]
        a += time.perf_counter() - t
        t = time.perf_counter()
        _ = pd.read_csv(path)
        r += time.perf_counter() - t
    n = len(target_ids)
    return g / n * 1000, a / n * 1000, r / n * 1000


id_cold = breakdown_by_id(clear=True)  # .artifacts() decompresses from the pool
id_warm = breakdown_by_id(clear=False)  # .artifacts() is a cache hit
for label, (g, a, r) in [("COLD (cache miss)", id_cold), ("WARM (cache hit)", id_warm)]:
    print(
        f"{label:<20}  get_node {g:7.3f}  | artifacts() {a:7.3f}  | read_csv {r:7.3f}  | TOTAL {g + a + r:7.3f} ms"
    )

COLD (cache miss)     get_node   0.100  | artifacts()   0.167  | read_csv  12.470  | TOTAL  12.737 ms
WARM (cache hit)      get_node   0.074  | artifacts()   0.130  | read_csv  12.240  | TOTAL  12.444 ms


## 4. Retrieve through the store, by metadata search — every step

The realistic case where you *don't* know the id: `find_node(run_id=...)` scans the index, then the same discover + read. Shows what the search itself adds on top of section 3.

In [5]:
def breakdown_by_search(clear):
    if clear:
        store.clear_cache()
    s = a = r = 0.0
    for i, _ in enumerate(target_ids):
        t = time.perf_counter()
        node = store.find_node(run_id=i)[0]
        s += time.perf_counter() - t
        t = time.perf_counter()
        path = node.artifacts("data.csv")[0]
        a += time.perf_counter() - t
        t = time.perf_counter()
        _ = pd.read_csv(path)
        r += time.perf_counter() - t
    n = len(target_ids)
    return s / n * 1000, a / n * 1000, r / n * 1000


se_cold = breakdown_by_search(clear=True)
se_warm = breakdown_by_search(clear=False)
for label, (s, a, r) in [("COLD", se_cold), ("WARM", se_warm)]:
    print(
        f"{label:<6} find_node {s:7.3f}  | artifacts() {a:7.3f}  | read_csv {r:7.3f}  | TOTAL {s + a + r:7.3f} ms"
    )

COLD   find_node   0.221  | artifacts()   0.169  | read_csv  12.128  | TOTAL  12.519 ms
WARM   find_node   0.215  | artifacts()   0.147  | read_csv  12.049  | TOTAL  12.410 ms


## 5. The convenience path: `node / "data.csv"`

When you know the filename you skip `.artifacts()` and index it directly. Same materialization underneath.

In [6]:
def by_slash(clear):
    if clear:
        store.clear_cache()
    tot = 0.0
    for tid in target_ids:
        t = time.perf_counter()
        _ = pd.read_csv(store.get_node(tid) / "data.csv")
        tot += time.perf_counter() - t
    return tot / len(target_ids) * 1000


print(
    f"node / 'data.csv' then read  COLD {by_slash(True):7.3f} ms   WARM {by_slash(False):7.3f} ms"
)

node / 'data.csv' then read  COLD  12.366 ms   WARM  12.303 ms


## 6. Summary — store overhead over a direct path read

In [7]:
def row(name, total):
    print(f"  {name:<34} {total:8.3f} ms   ({total / direct_parse:5.1f}x direct)")


print(f"Reading one ~{art_mb:.1f} MB CSV ({len(target_ids)} samples averaged)\n")
print(f"  {'direct pd.read_csv(path)':<34} {direct_parse:8.3f} ms   (baseline)")
row("store by id   — COLD (first touch)", sum(id_cold))
row("store by id   — WARM (cache hit)", sum(id_warm))
row("store by search — COLD", sum(se_cold))
row("store by search — WARM", sum(se_warm))
print()
print(
    f"  decompression cost (artifacts COLD - WARM): {id_cold[1] - id_warm[1]:.3f} ms / artifact"
)
print(
    f"  index search cost  (find_node):             {se_cold[0]:.3f} ms over {N_TOTAL} nodes"
)
print(f"  meta.json load     (get_node):              {id_cold[0]:.3f} ms")
print()
print("Takeaways:")
print(
    " - WARM store reads ~= a direct read: once cached, the store adds almost nothing."
)
print(
    " - The COLD premium is the chunk decompression, paid once per artifact per session."
)
print(
    " - Finding a node (id load or index search) is sub-millisecond and dwarfed by I/O."
)

Reading one ~2.2 MB CSV (30 samples averaged)

  direct pd.read_csv(path)             12.178 ms   (baseline)
  store by id   — COLD (first touch)   12.737 ms   (  1.0x direct)
  store by id   — WARM (cache hit)     12.444 ms   (  1.0x direct)
  store by search — COLD               12.519 ms   (  1.0x direct)
  store by search — WARM               12.410 ms   (  1.0x direct)

  decompression cost (artifacts COLD - WARM): 0.037 ms / artifact
  index search cost  (find_node):             0.221 ms over 1000 nodes
  meta.json load     (get_node):              0.100 ms

Takeaways:
 - WARM store reads ~= a direct read: once cached, the store adds almost nothing.
 - The COLD premium is the chunk decompression, paid once per artifact per session.
 - Finding a node (id load or index search) is sub-millisecond and dwarfed by I/O.


## 7. Read/write penalty by data type and size

The headline table: for a spread of file types and sizes, how long does it take to **write** and **read** the file *directly* (a plain path) versus *through the store* (chunked into the pool on write; reassembled from the pool into the read cache on a cold read; a plain cache read when warm)?

- **W store** = the loose-file write **+** packing it into the chunk pool. It excludes node creation (a fixed ~16 ms of git provenance — a separate issue), so this is purely the storage-layer penalty on the bytes.
- **R cold** = first touch in a session (decompress every chunk → cache).
- **R warm** = a second read in the same session (cache hit).

All `x` columns are the multiple over the *direct* operation. The largest rows take a few seconds (packing thousands of chunks).

In [8]:
import shutil
import time
import pickle
from pathlib import Path
import numpy as np
import ancestree

DT_ROOT = Path("dt_bench_store")
DT_BASE = Path("dt_bench_direct")
for d in (DT_ROOT, DT_BASE):
    if d.exists():
        shutil.rmtree(d)
DT_BASE.mkdir(parents=True)
dt_store = ancestree.LineageStore(DT_ROOT, dedupe=False, chunk=True)
dt_rng = np.random.default_rng(0)


def _med(fn, reps):
    ts = []
    for _ in range(reps):
        t = time.perf_counter()
        fn()
        ts.append((time.perf_counter() - t) * 1000)
    return float(np.median(ts))


def _npy_w(p, d):
    with open(p, "wb") as f:
        np.save(f, d)


def _npy_r(p):
    with open(p, "rb") as f:
        return np.load(f)


def _npz_w(p, d):
    with open(p, "wb") as f:
        np.savez_compressed(f, a=d)


def _npz_r(p):
    with open(p, "rb") as f:
        return np.load(f)["a"]


FORMATS = {
    "bytes": dict(
        fn="d.bin",
        make=lambda a: a.tobytes(),
        w=lambda p, d: Path(p).write_bytes(d),
        r=lambda p: Path(p).read_bytes(),
    ),
    "npy": dict(fn="d.npy", make=lambda a: a, w=_npy_w, r=_npy_r),
    "npz": dict(fn="d.npz", make=lambda a: a, w=_npz_w, r=_npz_r),
    "csv": dict(
        fn="d.csv",
        make=lambda a: pd.DataFrame(
            a[: len(a) // 4 * 4].reshape(-1, 4), columns=list("abcd")
        ),
        w=lambda p, d: d.to_csv(p, index=False),
        r=lambda p: pd.read_csv(p),
    ),
    "pickle": dict(
        fn="d.pkl",
        make=lambda a: a,
        w=lambda p, d: Path(p).write_bytes(pickle.dumps(d)),
        r=lambda p: pickle.loads(Path(p).read_bytes()),
    ),
}
SIZES_MB = [
    0.25,
    1,
    4,
    16,
    64,
]  # underlying float64 array size; file sizes differ per format

rows = []
for size_mb in SIZES_MB:
    arr = dt_rng.standard_normal(int(size_mb * 1e6 / 8))
    for name, F in FORMATS.items():
        data = F["make"](arr)
        dpath = DT_BASE / f"{name}_{size_mb}_{F['fn']}"
        wd = _med(lambda: F["w"](dpath, data), 5)  # direct write
        file_mb = dpath.stat().st_size / 1e6
        rd = _med(lambda: F["r"](dpath), 7)  # direct read
        with dt_store.create_node(
            step_type="blob"
        ) as node:  # store write = write + pack
            F["w"](node / F["fn"], data)
            t0 = time.perf_counter()
            node._pack()
            pack = (time.perf_counter() - t0) * 1000
        nid = node.node_id

        def _cold():
            dt_store.clear_cache()
            return F["r"](dt_store.get_node(nid) / F["fn"])

        rc = _med(_cold, 5)  # cold read (materialize + parse)
        rw = _med(
            lambda: F["r"](dt_store.get_node(nid) / F["fn"]), 7
        )  # warm read (cache hit)
        rows.append(
            dict(
                type=name,
                file_mb=file_mb,
                wd=wd,
                pack=pack,
                ws=wd + pack,
                rd=rd,
                rc=rc,
                rw=rw,
            )
        )
    print(f"  done size {size_mb} MB")
print("measurements complete")

  done size 0.25 MB
  done size 1 MB
  done size 4 MB
  done size 16 MB
  done size 64 MB
measurements complete


### The table

In [9]:
hdr = (
    f"{'type':7} {'file MB':>8} | {'W direct':>9} {'pack':>8} {'W store':>9} {'W x':>6} | "
    f"{'R direct':>9} {'R cold':>8} {'R warm':>8} {'cold x':>7} {'warm x':>7}"
)
print(hdr)
print("-" * len(hdr))
for r in rows:
    print(
        f"{r['type']:7} {r['file_mb']:8.2f} | {r['wd']:9.2f} {r['pack']:8.2f} {r['ws']:9.2f} "
        f"{r['ws'] / r['wd']:5.0f}x | {r['rd']:9.2f} {r['rc']:8.2f} {r['rw']:8.2f} "
        f"{r['rc'] / r['rd']:6.0f}x {r['rw'] / r['rd']:6.1f}x"
    )

print(
    "\nReadings are milliseconds (median). W store excludes ~16 ms/node of git provenance."
)

type     file MB |  W direct     pack   W store    W x |  R direct   R cold   R warm  cold x  warm x
----------------------------------------------------------------------------------------------------
bytes       0.25 |      0.14    27.48     27.61   202x |      0.03     0.21     0.15      8x    5.7x
npy         0.25 |      0.11    23.10     23.21   207x |      0.04     0.27     0.22      6x    4.9x
npz         0.24 |      5.65    24.60     30.26     5x |      0.59     0.77     0.70      1x    1.2x
csv         0.61 |     17.80    74.12     91.92     5x |      2.24     2.40     2.31      1x    1.0x
pickle      0.25 |      0.11    22.55     22.66   209x |      0.03     0.19     0.15      6x    4.8x
bytes       1.00 |      0.22   110.11    110.33   495x |      0.05     0.28     0.21      6x    4.7x
npy         1.00 |      0.26    89.31     89.56   351x |      0.07     0.29     0.23      4x    3.4x
npz         0.96 |     23.63   102.73    126.36     5x |      2.10     2.32     2.25      1

### Takeaways

In [10]:
fast = [r for r in rows if r["type"] in ("bytes", "npy", "pickle")]
slow = [r for r in rows if r["type"] == "csv"]
print("WRITE penalty is the big one:")
print(
    f"  packing turns a ~{min(r['wd'] for r in fast):.2f}-{max(r['wd'] for r in fast):.1f} ms direct write into"
)
print(
    f"  {min(r['ws'] for r in fast):.0f}-{max(r['ws'] for r in fast):.0f} ms — up to {max(r['ws'] / r['wd'] for r in fast):.0f}x — because each ~32 KB chunk"
)
print(
    "  is a separate compress+temp-write+rename. Bigger artifacts pay proportionally more."
)
print()
print("READ penalty depends entirely on the format:")
print(
    f"  fast binary (npy/bytes/pickle): cold read up to {max(r['rc'] / r['rd'] for r in fast):.0f}x direct"
)
print("     (the decompression dwarfs the ~0-1 ms raw read), warm ~1x once cached.")
print(
    f"  slow text (csv): cold only ~{max(r['rc'] / r['rd'] for r in slow):.1f}x — the parse hides the decompression."
)
print()
print("So: the store is cheap when your I/O is already slow (csv), and expensive")
print("relative to fast formats (npy) — especially on WRITE. Warm re-reads are ~free.")

shutil.rmtree(DT_ROOT, ignore_errors=True)
shutil.rmtree(DT_BASE, ignore_errors=True)

WRITE penalty is the big one:
  packing turns a ~0.11-12.0 ms direct write into
  23-7063 ms — up to 937x — because each ~32 KB chunk
  is a separate compress+temp-write+rename. Bigger artifacts pay proportionally more.

READ penalty depends entirely on the format:
  fast binary (npy/bytes/pickle): cold read up to 8x direct
     (the decompression dwarfs the ~0-1 ms raw read), warm ~1x once cached.
  slow text (csv): cold only ~1.1x — the parse hides the decompression.

So: the store is cheap when your I/O is already slow (csv), and expensive
relative to fast formats (npy) — especially on WRITE. Warm re-reads are ~free.


## 8. Deferred packing — what the write path *actually* costs now

Section 7's **W store** column packed the artifact synchronously, on the write path. Packing is now **deferred**: `create_node` writes the artifact as a normal file and returns; a background worker chunks it off the write path. This cell times the real `create_node` block (**W deferred**) against a direct write, and separately surfaces the **background pack** cost that `flush()` blocks on — the work that used to be section 7's **W store**, now moved off the critical path.

Reuses `FORMATS`, `_med`, `dt_rng` from section 7, so run that first. `dedupe=False` here so the table isolates the storage write path; with `dedupe=True` the block also does a content-hash pass that scales with size.

In [11]:
import shutil
import time
from pathlib import Path
import ancestree

DEF_STORE = Path("deferred_store")
DEF_DIRECT = Path("deferred_direct")
for d in (DEF_STORE, DEF_DIRECT):
    if d.exists():
        shutil.rmtree(d)
DEF_DIRECT.mkdir(parents=True)
dstore = ancestree.LineageStore(DEF_STORE, dedupe=False, chunk=True)

DEFER_SIZES = [1, 16, 64]  # MB; a representative span (packing scales with size)
hdr = (
    f"{'type':7} {'file MB':>8} | {'W direct':>9} {'W deferred':>11} {'defer x':>8} "
    f"| {'bg pack':>9} {'(was W store)':>14}"
)
print(hdr)
print("-" * len(hdr))
for size_mb in DEFER_SIZES:
    arr = dt_rng.standard_normal(int(size_mb * 1e6 / 8))
    for name, F in FORMATS.items():
        data = F["make"](arr)
        dpath = DEF_DIRECT / f"{name}_{size_mb}_{F['fn']}"
        wd = _med(lambda: F["w"](dpath, data), 5)  # direct write
        file_mb = dpath.stat().st_size / 1e6
        # Store write through the real API: native write + enqueue, returns now.
        t0 = time.perf_counter()
        with dstore.create_node(step_type="blob") as node:
            F["w"](node / F["fn"], data)
        w_defer = (time.perf_counter() - t0) * 1000
        # The chunking the worker does off-path; flush() blocks until it's done.
        t0 = time.perf_counter()
        dstore.flush()
        bg_pack = (time.perf_counter() - t0) * 1000
        print(
            f"{name:7} {file_mb:8.2f} | {wd:9.2f} {w_defer:11.2f} {w_defer / wd:6.1f}x "
            f"| {bg_pack:9.1f} {'(off path)':>14}"
        )

print("\nW deferred = the create_node block you actually wait for (native write +")
print("enqueue + ~16 ms/node provenance). bg pack = chunking moved to the background")
print("worker (section 7 paid this ON the write path). The write LATENCY is what")
print("dropped; the packing still happens, just off the critical path.")
shutil.rmtree(DEF_STORE, ignore_errors=True)
shutil.rmtree(DEF_DIRECT, ignore_errors=True)

type     file MB |  W direct  W deferred  defer x |   bg pack  (was W store)
----------------------------------------------------------------------------
bytes       1.00 |      0.23       29.09  129.2x |     109.3     (off path)
npy         1.00 |      0.24       27.94  115.2x |      83.5     (off path)
npz         0.96 |     23.51       49.10    2.1x |      97.4     (off path)
csv         2.45 |     72.04       96.62    1.3x |     290.9     (off path)
pickle      1.00 |      0.32       25.71   80.6x |      84.1     (off path)
bytes      16.00 |      2.41       27.34   11.4x |    1762.0     (off path)
npy        16.00 |      2.58       34.86   13.5x |    1344.3     (off path)
npz        15.37 |    375.32      400.12    1.1x |    1501.4     (off path)
csv        39.26 |   1102.39     1134.24    1.0x |    4641.5     (off path)
pickle     16.00 |      2.99       30.19   10.1x |    1314.3     (off path)
bytes      64.00 |      7.47       31.86    4.3x |    6743.3     (off path)
npy       